# Multivariate EDA: looking at many variables and groups at once

_Data Wrangling_

**Compare segments and high-dimensional structure — and never trust an aggregate before you check the groups.**

Looking at one column at a time tells you the shape of each variable. Looking at two
       columns together (a scatter, a correlation) tells you how a pair moves. Multivariate
       Exploratory Data Analysis (EDA) is the next step up: looking at many variables &mdash; and
       crucially many groups or segments &mdash; at the same time.

       You have a small toolbox for this. The pair plot (also called a scatter-matrix) draws every
       pair of features as a grid of scatter plots, so you scan all the two-way relationships in one glance.
       Encodings &mdash; color, marker size, and faceting (small multiples) &mdash; let you fold extra
       variables into a single chart. Grouped aggregation (groupby().agg() in pandas)
       collapses each segment to a few summary numbers so you can compare them in a table. And when there are
       too many features to plot pairwise, you project the data down to 2-D with
       PCA (Principal Component Analysis) or
       t-SNE and look at the cloud.

---

This notebook builds the multivariate-EDA toolbox one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. It's a real table — here are its columns and the first few rows.

In [ ]:
import seaborn as sns

data = sns.load_dataset("tips")
print("rows x columns:", data.shape)
print("columns:", list(data.columns))
data.head()

## Reference implementation — pandas + seaborn

We build the multivariate toolbox in three steps: (1) compare segments with grouped aggregation, (2) scan every pair of features with a color-coded pair plot, and (3) watch a per-group trend reverse when pooled — Simpson's paradox on a real trial.

### Step 1 — Compare segments with groupby().agg()

Grouped aggregation collapses each segment to a few summary numbers so you can compare them in a table. Here one row is a `(day, smoker)` segment, and we compute the count, the mean bill, the mean tip, and the mean tip percentage. Reading down the table compares segments directly.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")          # bundled with seaborn

# One row per (day, smoker) segment, several summaries each.
seg = (tips
       .groupby(["day", "smoker"])
       .agg(n=("total_bill", "size"),
            mean_bill=("total_bill", "mean"),
            mean_tip=("tip", "mean"),
            tip_pct=("tip", lambda s: (s / tips.loc[s.index, "total_bill"]).mean()))
       .round(2))
print(seg)
# Compare segments by reading down the table -- e.g. tip % across days/smokers.

### Step 2 — Scan every feature pair with a pair plot

The pair plot (scatter-matrix) draws every pair of features as a grid of scatter plots, so you scan all the two-way relationships at once. Color (`hue`) folds in a third variable — here lunch vs. dinner. For high-dimensional data this grid explodes to $k^2$ panels, so you project with PCA or t-SNE instead.

In [ ]:
# Every pair of three features, with a 3rd variable folded in via COLOR (hue).
sns.pairplot(tips, vars=["total_bill", "tip", "size"], hue="time",
             diag_kind="hist", height=2.2)
plt.suptitle("tips: scatter-matrix colored by lunch/dinner", y=1.02)
plt.show()

# For HIGH-dimensional data a pairplot explodes (k^2 panels); project instead:
#   from sklearn.decomposition import PCA   # see lesson fe-pca
#   from sklearn.manifold import TSNE       # see lesson cls-tsne
#   XY = PCA(n_components=2).fit_transform(X)   # then scatter XY colored by group

### Step 3 — Watch a trend reverse: Simpson's paradox

We rebuild a real kidney-stone trial (Charig et al., 1986) row by row from its `(successes, total)` cells. Treatment A has the higher success rate within **both** stone-size groups, yet B wins once the groups are pooled — because A was given mostly to the harder large stones. The lesson: always check whether an aggregate survives within groups.

In [ ]:
# Real kidney-stone trial. Each cell is (successes, total) for a
# treatment x stone-size combination.
cells = {("A", "small"): (81, 87),  ("A", "large"): (192, 263),
         ("B", "small"): (234, 270), ("B", "large"): (55, 80)}

# Expand the counts into one row per patient.
rows = []
for (trt, stone), (succ, n) in cells.items():
    rows += [(trt, stone, 1)] * succ + [(trt, stone, 0)] * (n - succ)
kidney = pd.DataFrame(rows, columns=["treatment", "stone_size", "success"])

# Per-group success rate: A wins for BOTH small and large stones.
per_group = (kidney.groupby(["stone_size", "treatment"])["success"]
             .mean().mul(100).round(1).unstack())
print("Per-group success rate (%):")
print(per_group)          # small: A 93.1 > B 86.7 ; large: A 73.0 > B 68.8

# Pooled success rate: B wins overall -- the trend REVERSES.
overall = kidney.groupby("treatment")["success"].mean().mul(100).round(1)
print("\nOverall success rate (%):")
print(overall)            # A 78.0  vs  B 82.6   <-- pooled winner is B!

# Why: A was given mostly to the HARDER large stones, dragging its pooled rate.
group_sizes = kidney.groupby(["treatment", "stone_size"]).size().unstack()
print("\nGroup sizes (rows per treatment x stone_size):")
print(group_sizes)
# Lesson: always check whether an aggregate relationship survives WITHIN groups.

## Visualize the data & results

_On the real kidney-stone trial, treatment A beats B for SMALL stones and for LARGE stones — does A still win once the two groups are pooled?_

### Step 1 — Rebuild the trial and compute per-group rates

We expand the same `(successes, total)` cells into one row per patient, then take the per-group success rate. Within each stone-size group, treatment A comes out ahead — the like-for-like comparison.

In [ ]:
import pandas as pd

# Charig et al. (1986) kidney-stone trial. Each cell = (successes, total)
# for one treatment x stone-size combination.
cells = {("A", "small"): (81, 87),  ("A", "large"): (192, 263),
         ("B", "small"): (234, 270), ("B", "large"): (55, 80)}

# Expand the counts into one row per patient.
rows = []
for (trt, stone), (succ, n) in cells.items():
    rows += [(trt, stone, 1)] * succ + [(trt, stone, 0)] * (n - succ)
df = pd.DataFrame(rows, columns=["treatment", "stone_size", "success"])

# Per-group rates -> A wins both groups.
per = df.groupby(["stone_size", "treatment"])["success"].mean().mul(100)
print("small A=%.1f  B=%.1f" % (per["small", "A"], per["small", "B"]))  # 93.1 86.7
print("large A=%.1f  B=%.1f" % (per["large", "A"], per["large", "B"]))  # 73.0 68.8

### Step 2 — Pool the groups and see the reversal

Now collapse the stone-size split and take the overall rate per treatment. Treatment B wins the pooled comparison even though it lost both subgroups — the reversal you would chart side by side against Step 1.

In [ ]:
# Pooled rates -> B wins overall: the reversal.
overall = df.groupby("treatment")["success"].mean().mul(100)
print("ALL   A=%.1f  B=%.1f" % (overall["A"], overall["B"]))           # 78.0 82.6

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Treatment A beats treatment B for small stones (93% vs 87%) and for large stones (73% vs 69%), yet B has the higher overall success rate (83% vs 78%). Which treatment should a patient prefer, and what produced the reversal?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that A wins inside every subgroup. — _The per-group rates are the like-for-like comparison: same stone size, so the only difference is the treatment._
- Look at how cases were assigned to each treatment. — _A was given mostly to large (harder) stones; B mostly to small (easier) ones, so the groups are not comparable in the pool._
- Recognize the pooled rate is a size-weighted average, dragged by A's heavy large-stone load. — _Unequal weights on unequal group rates make the aggregate flip even though no per-group rate changed._

**Answer:** Prefer treatment A: it wins in both stone-size groups, which is the fair, like-for-like comparison. The overall reversal is Simpson's paradox, caused by confounding &mdash; A was disproportionately used on the harder large stones, so its pooled rate is pulled down. The per-group result is the trustworthy one; the aggregate is misleading.

</details>

**Problem 2.** You have 30 numeric features and want to see the multivariate structure. A teammate suggests a seaborn pairplot of all 30. Why is that a bad idea, and what are two better options?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the panels a pairplot would draw. — _A pairplot of $k$ features is a $k \times k$ grid; for $k=30$ that is 900 tiny scatter plots, far too many to read._
- Option 1: pick a handful of the most relevant features and pairplot only those. — _A 4-to-6 feature pairplot stays legible and still shows the key two-way relationships._
- Option 2: project all 30 features to 2-D with PCA or t-SNE and plot the single cloud. — _A projection scales to high dimension and shows global structure (clusters, gradients, outliers) in one chart._

**Answer:** A 30-feature pairplot is $30^2 = 900$ panels &mdash; it explodes and is unreadable. Better: (1) pairplot a small chosen subset of features so each panel is legible, or (2) project to 2-D with PCA or t-SNE and look at the overall cloud. With t-SNE, remember its cluster sizes and gaps are not literally meaningful, so confirm any apparent clusters another way.

</details>

**Problem 3.** A t-SNE plot of your embeddings shows five crisp, well-separated blobs, and a colleague concludes "there are exactly five natural clusters." What is the risk, and how would you check?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what t-SNE optimizes. — _t-SNE preserves local neighborhoods and can pull data into tight, well-separated blobs even when the underlying structure is continuous or has no real clusters._
- Treat the number, size, and separation of blobs as suspect. — _t-SNE cluster sizes and between-cluster distances are not meaningful, so "exactly five" is not evidence on its own._
- Cross-check with another method. — _A PCA view, different t-SNE perplexities, or an actual clustering metric (e.g. silhouette) tell you whether the five blobs are real._

**Answer:** The risk is reading a t-SNE artifact as real structure: t-SNE can manufacture crisp, separated blobs from data with no genuine clusters, and its blob sizes and gaps carry no quantitative meaning. Check by re-running at several perplexities, comparing against a PCA view, and validating with a real clustering criterion (e.g. silhouette score) before claiming "exactly five clusters."

</details>